# Linear Algebra Capstone — Content-Based Movie Recommendations

**Estimated focused time:** 90–120 minutes. **Prerequisites:** linear algebra Notebooks 01–03. This is the final project in this linear algebra sequence; no external dataset or new framework is needed.

## Learning objectives

Represent items as feature vectors, combine liked items into a user profile, use dot products and L2 norms for cosine ranking, evaluate recommendations with precision/recall at `k`, and explain scale and cold-start limitations.

## Just enough theory

An item-feature matrix has one movie per row and one content feature per column. Averaging liked item vectors gives a simple taste profile. Cosine similarity compares each candidate with that profile. `Precision@k` is the fraction of the top `k` recommendations that are relevant; `recall@k` is the fraction of held-out relevant items found in the top `k`. The relevance labels are **evaluation data**: never use them to build a profile, set weights, or choose a ranking.

Implement the starter functions yourself. Tests verify outcomes but do not provide finished function bodies. Use NumPy and Python only.

## Synthetic catalog and one user

The columns represent science fiction, mystery, action, family, and comedy tags. The user liked titles 0 and 2. The held-out relevance set is intentionally small: it supports practice with metrics, not claims about real-user performance.

In [ ]:
import numpy as np

titles = np.array(['Orbit', 'Harbor', 'Signal', 'Garden', 'Nebula',
                   'Laughter', 'Quest', 'Cipher', 'Home', 'Pulse'])
item_features = np.array([
    [1., 0., 0., 1., 0.], [0., 0., 0., 1., 1.],
    [1., 0., 1., 0., 0.], [0., 1., 0., 0., 1.],
    [1., 0., 1., 1., 0.], [0., 0., 0., 0., 1.],
    [0., 1., 1., 0., 0.], [1., 0., 0., 1., 1.],
    [0., 0., 0., 1., 0.], [1., 0., 1., 0., 1.],
])
feature_names = ('sci_fi', 'mystery', 'action', 'family', 'comedy')
liked_indices = np.array([0, 2])
relevant_indices = {4, 7, 9}  # held-out labels: use only in Task 3 evaluation
popularity = np.array([9, 5, 8, 4, 7, 3, 6, 5, 7, 4])
assert item_features.shape == (len(titles), len(feature_names)) == (10, 5)

## Task 1 — Build a taste profile (15–20 minutes)

**Goal:** Turn liked item rows into one validated user vector.

### Instructions

1. Implement `mean_profile(items, liked)` for a nonempty 2D numerical matrix and unique, valid liked indices. Return a length-`d` 1D mean vector.
2. Calculate the science-fiction coordinate by hand; identify the axes of `item_features`.
3. Reject empty, duplicate, out-of-range liked indices and a 1D item array.

**Expected output:** A five-value profile, with science-fiction coordinate `1.0`.

### Acceptance criteria

- [ ] The profile uses only liked rows, never relevance labels.
- [ ] Valid output matches the NumPy reference; invalid inputs raise `ValueError`.

**Hint:** The profile is an average across selected rows, not across feature columns.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def mean_profile(items, liked):
    # TODO: validate items and liked indices, then return one 1D profile.
    raise NotImplementedError

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
profile = mean_profile(item_features, liked_indices)
assert profile.shape == (5,)
np.testing.assert_allclose(profile, item_features[liked_indices].mean(axis=0))
np.testing.assert_allclose(profile[0], 1.)
for bad in ([], [0, 0], [len(titles)]):
    np.testing.assert_raises(ValueError, mean_profile, item_features, bad)
np.testing.assert_raises(ValueError, mean_profile, item_features[0], liked_indices)

## Task 2 — Score and rank unseen movies (25–30 minutes)

**Goal:** Use cosine similarity to return a deterministic top-k ranking.

### Instructions

1. Implement `cosine_scores(items, profile)` returning one finite score per row. A zero-feature row gets score `0`; a zero profile or feature-count mismatch raises `ValueError`.
2. Implement `top_k_unseen(scores, seen, k)` returning unseen indices sorted by descending score, then ascending index for ties. Reject invalid `k` and invalid/duplicate seen indices.
3. Report the top three titles and scores without looking at held-out relevance labels.

**Expected output:** A length-10 score vector and top-three indices `[4,7,9]` on the supplied catalog.

### Acceptance criteria

- [ ] Cosine agrees with a hand/reference score for one movie.
- [ ] Zero and shape edge cases pass.
- [ ] Liked movies are excluded; tie order is deterministic.

**Hint:** For a nonzero row, cosine uses its dot product with the profile divided by both L2 norms. Sort only after excluding seen items.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def cosine_scores(items, profile):
    # TODO: validate, compute cosine scores, and handle zero-feature rows.
    raise NotImplementedError

def top_k_unseen(scores, seen, k):
    # TODO: validate, exclude seen indices, sort with index tie-break, and return indices.
    raise NotImplementedError

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
scores = cosine_scores(item_features, profile)
assert scores.shape == (10,) and np.all(np.isfinite(scores))
reference_one = np.dot(item_features[4], profile) / (np.linalg.norm(item_features[4]) * np.linalg.norm(profile))
np.testing.assert_allclose(scores[4], reference_one, atol=1e-12)
np.testing.assert_allclose(cosine_scores(np.vstack((item_features, np.zeros(5))), profile)[-1], 0)
np.testing.assert_raises(ValueError, cosine_scores, item_features, np.zeros(5))
np.testing.assert_raises(ValueError, cosine_scores, item_features, np.ones(4))
top3 = np.asarray(top_k_unseen(scores, liked_indices, 3))
np.testing.assert_array_equal(top3, [4, 7, 9])
np.testing.assert_array_equal(top_k_unseen([.4, .4, .1], [], 2), [0, 1])
for seen, k in ((liked_indices, 0), (liked_indices, 9), ([0, 0], 2), ([99], 2)):
    np.testing.assert_raises(ValueError, top_k_unseen, scores, seen, k)

## Task 3 — Evaluate without leaking labels (20–25 minutes)

**Goal:** Compare the recommender with a simple popularity baseline on the same candidate pool.

### Instructions

1. Implement `precision_recall_at_k(recommended, relevant)` for a nonempty relevant set. Reject duplicate recommendations and an empty relevant set.
2. Compute precision@3 and recall@3 for `top3`. Build a popularity top-three list after excluding liked items; break popularity ties by smaller index.
3. Compare both results and explain why one synthetic user cannot establish broad model quality.

**Expected output:** Model metrics `(1.0,1.0)` and popularity baseline indices `[4,8,6]` with metrics `(1/3,1/3)`.

### Acceptance criteria

- [ ] Held-out labels are used only for evaluation.
- [ ] Metric and baseline assertions pass.
- [ ] Comparison mentions the small synthetic sample.

**Hint:** Both methods must rank only movies the user has not already liked.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
def precision_recall_at_k(recommended, relevant):
    # TODO: validate and return (precision_at_k, recall_at_k).
    raise NotImplementedError

# TODO: produce baseline_top3, model_metrics, baseline_metrics.
baseline_top3 = model_metrics = baseline_metrics = None

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
np.testing.assert_allclose(precision_recall_at_k([4, 1, 7], relevant_indices), [2/3, 2/3])
np.testing.assert_raises(ValueError, precision_recall_at_k, [4, 4], relevant_indices)
np.testing.assert_raises(ValueError, precision_recall_at_k, [4], set())
assert baseline_top3 is not None and model_metrics is not None and baseline_metrics is not None
np.testing.assert_array_equal(baseline_top3, [4, 8, 6])
np.testing.assert_allclose(model_metrics, [1., 1.])
np.testing.assert_allclose(baseline_metrics, [1/3, 1/3])

## Task 4 — Test a limitation and cold start (20–25 minutes)

**Goal:** Show how feature weighting affects results and define a fallback for a new user.

### Instructions

1. Copy `item_features`, multiply its **action** column (index 2) by 10, rebuild the profile, and recompute the top three. Do not mutate the original matrix or use relevance labels to choose the weight.
2. For a user with no likes, catch `mean_profile`'s `ValueError` and use popularity to recommend three titles.
3. In 3–5 sentences, explain why the weighted ranking changes, what the cold-start fallback sacrifices, and one limitation of this toy evaluation.

**Expected output:** Action-weighted top-three indices `[4,9,6]`; cold-start popularity indices `[0,2,4]`.

### Acceptance criteria

- [ ] Original data remain unchanged.
- [ ] Weighted and fallback rankings match expected values.
- [ ] Interpretation discusses units/weights and cold start without claiming production performance.

**Hint:** The cold-start user has no liked items, so the fallback cannot use a taste profile.

**Interpretation:** Write 2–3 sentences explaining what the result means and one limitation or ML consequence. Add your answer beneath this paragraph.

In [ ]:
# TODO: compute weighted_items, weighted_profile, weighted_top3, and cold_start_top3.
weighted_items = weighted_profile = weighted_top3 = cold_start_top3 = None

### Self-check

Run this cell after completing the starter code. Keep the expected values and assertions intact.

In [ ]:
assert weighted_items is not None and weighted_items.shape == item_features.shape
np.testing.assert_array_equal(item_features[:, 2], [0, 0, 1, 0, 1, 0, 1, 0, 0, 1])
np.testing.assert_allclose(weighted_items[:, 2], 10 * item_features[:, 2])
np.testing.assert_allclose(weighted_items[:, [0, 1, 3, 4]], item_features[:, [0, 1, 3, 4]])
np.testing.assert_allclose(weighted_profile, weighted_items[liked_indices].mean(axis=0))
np.testing.assert_array_equal(weighted_top3, [4, 9, 6])
np.testing.assert_raises(ValueError, mean_profile, item_features, [])
np.testing.assert_array_equal(cold_start_top3, [0, 2, 4])

## Completion checklist

- [ ] Every starter cell is implemented; no `TODO`, `None` placeholder, or `NotImplementedError` remains.
- [ ] Every self-check passes, including invalid-input tests.
- [ ] I can explain the hand-check and the ML interpretation in my own words.
- [ ] **Restart Kernel → Run All** succeeds in order, with no unexpected errors.
- [ ] I saved the notebook and recorded a meaningful Git commit.

**Conceptual exit questions**

1. What does the user profile represent mathematically?
2. Why does cosine use both the dot product and norms?
3. What do precision@3 and recall@3 measure here, and what do they fail to prove?
4. Why might a new user need a non-personalized fallback?

**Final deliverable:** A reproducible notebook containing a top-three recommendation table, baseline comparison, weighted-feature experiment, cold-start fallback, and a short interpretation.